In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

## Data Understanding

Tahapan ini dilakukan untuk menilai dataset dan sebagai eksplorasi awal, seperti mengecek ukuran dataset, memeriksa struktur data, mengecek missing value dan data duplikat. 

In [2]:
# Load dataset
df = pd.read_csv('dataset/spotify_songs.csv')
df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [3]:
# Melihat struktur data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32833 entries, 0 to 32832
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  32833 non-null  object 
 1   track_name                32828 non-null  object 
 2   track_artist              32828 non-null  object 
 3   track_popularity          32833 non-null  int64  
 4   track_album_id            32833 non-null  object 
 5   track_album_name          32828 non-null  object 
 6   track_album_release_date  32833 non-null  object 
 7   playlist_name             32833 non-null  object 
 8   playlist_id               32833 non-null  object 
 9   playlist_genre            32833 non-null  object 
 10  playlist_subgenre         32833 non-null  object 
 11  danceability              32833 non-null  float64
 12  energy                    32833 non-null  float64
 13  key                       32833 non-null  int64  
 14  loudne

In [4]:
# Mengecek jumlah data
df.shape

(32833, 23)

In [6]:
df.describe()

,track_popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
count,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000,32833.000000
mean,42.477081,0.654850,0.698619,5.374471,-6.719499,0.565711,0.107068,0.175334,0.084747,0.190176,0.510561,120.881132,225799.811622
std,24.984074,0.145085,0.180910,3.611657,2.988436,0.495671,0.101314,0.219633,0.224230,0.154317,0.233146,26.903624,59834.006182
min,0.000000,0.000000,0.000175,0.000000,-46.448000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4000.000000
25%,24.000000,0.563000,0.581000,2.000000,-8.171000,0.000000,0.041000,0.015100,0.000000,0.092700,0.331000,99.960000,187819.000000
50%,45.000000,0.672000,0.721000,6.000000,-6.166000,1.000000,0.062500,0.080400,0.000016,0.127000,0.512000,121.984000,216000.000000
75%,62.000000,0.761000,0.840000,9.000000,-4.645000,1.000000,0.132000,0.255000,0.004830,0.248000,0.693000,133.918000,253585.000000
max,100.000000,0.983000,1.000000,11.000000,1.275000,1.000000,0.918000,0.994000,0.994000,0.996000,0.991000,239.440000,517810.000000


In [7]:
# Mengecek jumlah missing value
df.isnull().sum()

track_id                    0
track_name                  5
track_artist                5
track_popularity            0
track_album_id              0
track_album_name            5
track_album_release_date    0
playlist_name               0
playlist_id                 0
playlist_genre              0
playlist_subgenre           0
danceability                0
energy                      0
key                         0
loudness                    0
mode                        0
speechiness                 0
acousticness                0
instrumentalness            0
liveness                    0
valence                     0
tempo                       0
duration_ms                 0
dtype: int64

In [8]:
print(f"Jumlah data lagu unik berdasarkan track_name: {df['track_name'].nunique()}")
print(f"Jumlah data lagu unik berdasarkan track_id: {df['track_id'].nunique()}")

Jumlah data lagu unik berdasarkan track_name: 23449
Jumlah data lagu unik berdasarkan track_id: 28356


In [11]:
duplicates = df[df.duplicated()]
print(f"Jumlah data duplikat: {len(duplicates)}")

Jumlah data duplikat: 0


**Insight**:
- terdapat 32833 baris dan 23 kolom
- terdapat missing value
- tidak ditemukan data duplikat
- ditemukan 28356 data unik berdasarkan track_id, kita perlu membersihkannya di tahap selanjutnya.

## Data Preparation

Tahapan data preparation dilakukan untuk memastikan bahwa data siap digunakan oleh model machine learning dan menghasilkan performa yang optimal

In [36]:
# Menghapus missing value
df.dropna(inplace=True)

In [23]:
df = df.drop_duplicates(subset='track_id').reset_index(drop=True)
print(f"Data setelah di-drop: {df.shape}")


Data setelah di-drop: (28352, 23)


In [17]:
# Menampilkan genre beserta sub genrenya
for genre in df['playlist_genre'].unique():
    print(f"{genre}: {df['playlist_subgenre'][df['playlist_genre'] == genre].unique().tolist()}")

pop: ['dance pop', 'post-teen pop', 'electropop', 'indie poptimism']
rap: ['hip hop', 'southern hip hop', 'gangster rap', 'trap']
rock: ['album rock', 'classic rock', 'permanent wave', 'hard rock']
latin: ['tropical', 'latin pop', 'reggaeton', 'latin hip hop']
r&b: ['urban contemporary', 'hip pop', 'new jack swing', 'neo soul']
edm: ['electro house', 'big room', 'pop edm', 'progressive electro house']


In [13]:
# Hitung rata-rata popularitas per album per artis
album_popularity = df.groupby(
    ['track_artist', 'track_album_id', 'track_album_name'], 
    as_index=False
)['track_popularity'].mean()

# Urutkan agar album paling populer per artis ada di atas
top_albums = album_popularity.sort_values(
    ['track_artist', 'track_popularity'], 
    ascending=[True, False]
)

# Ambil satu album paling populer untuk tiap artis
top_album_per_artist = top_albums.drop_duplicates('track_artist', keep='first')

# Tampilkan hasilnya
print(top_album_per_artist.head(10))


                                    track_artist          track_album_id  \
0                                            !!!  3ywERU9rVKEpCQE50fQOgM   
2                                       !deladap  3bmmclWhMybn1nh8leCkKb   
5   "Dear Evan Hansen" August 2018 Broadway Cast  2M1j4rQyLRsy91VpWTZ56k   
6                                   #TocoParaVos  4b10EvdYJG27bXYZEACjTy   
8                                          $ANFI  4BsrreTOIecyV2z0apFumw   
9                                          $IFRA  6iA9xPlfctyfDCdlaOD9ky   
15                                   $uicideBoy$  2zQsirZE9McFbEoLkhxcpk   
26                                  'Til Tuesday  1y4s0JN8CQMRwPTJ64jCUp   
27                                      (G)I-DLE  4VLR1cDqRIeS86GYSJvlmZ   
28                                (Sandy) Alex G  2kCDZ3gCr5hXFgbFsPMcxP   

                             track_album_name  track_popularity  
0                                    Thr!!!er              50.0  
2   Music Has the Power (Wolfga

Memilih fitur-fitur penting untuk sistem rekomendasi, termasuk track_name yang nantinya dijadikan index. Tahapan ini perlu karna kita ingin merekomendasikan lagu berdasarkan karakteristiknya.

In [ ]:
def prepare_features(df, text_col='track_name', drop_dupes=True):
    """
    Siapkan data fitur numerik untuk content-based recommendation.

    Parameters:
        df (DataFrame): Dataset asli.
        text_col (str): Kolom teks sebagai nama item (index).
        drop_dupes (bool): Apakah mau buang duplikat berdasarkan nama item.

    Returns:
        DataFrame: Data fitur numerik yang siap dipakai, index = nama item.
        list: Daftar fitur numerik yang digunakan.
    """
    features = ['danceability', 'energy',
                'loudness', 'speechiness', 'acousticness',
                'instrumentalness', 'liveness', 'valence', 
                'tempo', 'track_popularity']
    
    temp = df[[*features, text_col]].copy()

    if drop_dupes:
        temp.drop_duplicates(subset=text_col, inplace=True)

    temp.set_index(text_col, inplace=True)

    return temp, features


In [26]:
# Siapin data fitur numerik
temp, features = prepare_features(df)


Normalisasi diperlukan untuk memastikan semua fitur numerik berada di skala yang sama.

In [ ]:
# Normalisasi data numerik
scaler = MinMaxScaler()
scaled_numeric = scaler.fit_transform(temp)


## Modeling

Kolom track_name diubah ke numerik dengan algoritma TF-IDF karna model hanya bisa memproses numerik. algoritma ini merupakan teknik dari NLP yang digunakan untuk menghitung bobot kata-kata dalam judul lagu berdasarkan frekuensi kemunculannya, memungkinkan model untuk mengenali kemiripan antar lagu berdasarkan teks judulnya.

In [ ]:
# TF-IDF dari track_name
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(temp.index)  # karena track_name udah jadi index


In [ ]:
# Gabungin fitur numerik dan TF-IDF
combined_features = hstack([scaled_numeric, tfidf_matrix])


cosine similarity digunakan untuk mengukur kemiripan antar lagu berdasarkan vektor hasil tf-idf.

In [ ]:
# Hitung cosine similarity
sim_df = pd.DataFrame(
    cosine_similarity(combined_features),
    index=temp.index,
    columns=temp.index
)


## Evaluation

Fungsi di bawah akan menampilkan top 10 rekomendasi lagu teratas.

In [31]:
def recommend(track_name, sim_df, top_n=5):
    if track_name not in sim_df.index:
        return f"Lagu '{track_name}' tidak ditemukan."
    similarity_scores = sim_df[track_name].sort_values(ascending=False)
    return similarity_scores.iloc[1:top_n+1]  # skip urutan pertama (karena itu dirinya sendiri)


In [35]:
recommend('Closer', sim_df, top_n=10)


track_name
Closer Still             0.984355
Closer Than This         0.941774
Get Closer               0.910857
Closer - R3hab Remix     0.898034
One Step Closer          0.874817
Closer - T-Mass Remix    0.871476
Closer (Ode 2 U)         0.868300
Closer To The Heart      0.856007
Closer (feat. Halsey)    0.833029
SO AM I                  0.829545
Name: Closer, dtype: float64